In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("all-MiniLM-L6-v2")
import string
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
trans_table = str.maketrans('', '', string.punctuation)

In [ ]:
raw_verses = pd.read_csv('hebrew_numbering_claude_translation.csv')

In [ ]:
raw_translated_hebrew = raw_verses['Hebrew Trans']
raw_translated_septuagint = raw_verses['Septuagint Trans']
raw_translated_gallican_psalter = raw_verses['Gallican Psalter Trans']
raw_translated_hebrew_psalter = raw_verses['Hebrew Psalter Trans']

In [ ]:
def clean_score(score):
    return round(float(score), 5)

def compute_similarity(list1, list2, list3, list4):
    scores_1_2 = []
    scores_1_3 = []
    scores_1_4 = []
    scores_2_3 = []
    scores_2_4 = []
    scores_3_4 = []
    count = 0
    for raw_verse1, raw_verse2, raw_verse3, raw_verse4 in zip(list1, list2, list3, list4):

        verse1 = raw_verse1.translate(trans_table)
        verse2 = raw_verse2.translate(trans_table)
        verse3 = raw_verse3.translate(trans_table)
        verse4 = raw_verse4.translate(trans_table)


        embeddings = model.encode([verse1, verse2, verse3, verse4])
        similarities = model.similarity(embeddings, embeddings)
        scores_1_2.append(clean_score(similarities[1][0]))
        scores_1_3.append(clean_score(similarities[2][0]))
        scores_1_4.append(clean_score(similarities[3][0]))
        scores_2_3.append(clean_score(similarities[2][1]))
        scores_2_4.append(clean_score(similarities[3][1]))
        scores_3_4.append(clean_score(similarities[3][2]))
        count+=1
        if (count % 50 == 0):
            print(count)

    return scores_1_2, scores_1_3, scores_1_4, scores_2_3, scores_2_4, scores_3_4

In [ ]:
heb_sept,heb_pgal,heb_pheb,sept_pgal,sept_pheb,pgal_pheb = compute_similarity(
    raw_translated_hebrew, raw_translated_septuagint, raw_translated_gallican_psalter, raw_translated_hebrew_psalter)

In [ ]:
raw_verses['heb_sept'] = heb_sept
raw_verses['heb_pgal'] = heb_pgal
raw_verses['heb_pheb'] = heb_pheb
raw_verses['sept_pgal'] = sept_pgal
raw_verses['sept_pheb'] = sept_pheb
raw_verses['pgal_pheb'] = pgal_pheb

In [ ]:
normalized = []
for col in raw_verses.columns[-6:]:
    array = np.array(raw_verses[col])
    array = (array - array.min()) / (array.max() - array.min())
    array = array.round(5)
    #array = array/array.mean()
    raw_verses['norm_'+col] = array

In [ ]:
raw_stats = raw_verses[['heb_sept','heb_pgal','heb_pheb','sept_pgal','sept_pheb','pgal_pheb']]

In [ ]:
bins2 = []
for i in range(21):
    bins2.append(round(0.05*i,2))

names = {
    'heb_sept' : 'Hebrew Bible/Septuagint',
    'heb_pgal' : 'Hebrew Bible/Gallican Psalter',
    'heb_pheb' : 'Hebrew Bible/Hebrew Psalter ',
    'sept_pgal' : 'Septuagint/Gallican Psalter',
    'sept_pheb' : 'Septuagint/Hebrew Psalter',
    'pgal_pheb' : 'Gallican Psalter/Hebrew Psalter'
}

fig, axs = plt.subplots(2, 3)
axs[0, 0].hist(raw_stats['heb_sept'], bins = bins2[6:])
axs[0, 0].set_title(names['heb_sept'])
axs[0, 1].hist(raw_stats['heb_pgal'], bins = bins2[6:])
axs[0, 1].set_title(names['heb_pgal'])
axs[0, 2].hist(raw_stats['heb_pheb'], bins = bins2[6:])
axs[0, 2].set_title(names['heb_pheb'])
axs[1, 0].hist(raw_stats['sept_pgal'], bins = bins2[6:])
axs[1, 0].set_title(names['sept_pgal'])
axs[1, 1].hist(raw_stats['sept_pheb'], bins = bins2[6:])
axs[1, 1].set_title(names['sept_pheb'])
axs[1, 2].hist(raw_stats['pgal_pheb'], bins = bins2[6:])
axs[1, 2].set_title(names['pgal_pheb'])



for ax in axs.flat:
    ax.set(xlabel='Cosine Similarity', ylabel='Number of Verses')
    ax.set_ylim(0,1000)


for ax in axs.flat:
    ax.label_outer()

plt.subplots_adjust(right = 1.3)
fig.suptitle('Machine Translation - Distribution of Cosine Similarities by Text Pairing', x=0.75, fontsize = 13)